In [2]:
import backtrader as bt
import pandas as pd
import numpy as np
import datetime
from copy import deepcopy

# 一、读取日度行情表
表内字段就是Backtrader默认情况下要求输入的7个字段：'datetime'、'open'、'high'、'low'、'close'、'volume'、'openinterest'，外加一个'sec_code'股票代码字段

In [3]:
daily_price = pd.read_csv("./data/daily_price.csv", parse_dates=['datetime'])
daily_price

,datetime,sec_code,open,high,low,close,volume,openinterest
0,2019-01-02,600466.SH,33.064891,33.496709,31.954503,32.386321,10629352,0
1,2019-01-02,603228.SH,50.660230,51.458513,50.394136,51.120778,426147,0
2,2019-01-02,600315.SH,148.258423,150.480132,148.258423,149.558935,2138556,0
3,2019-01-02,000750.SZ,49.512579,53.154883,48.715825,51.561375,227557612,0
4,2019-01-02,002588.SZ,36.608672,36.608672,35.669988,35.763857,2841517,0
...,...,...,...,...,...,...,...,...
255967,2021-01-28,600717.SH,121.489201,122.011736,120.705400,120.966667,6022213,0
255968,2021-01-28,300558.SZ,134.155888,137.600704,130.700970,131.569750,5330301,0
255969,2021-01-28,600171.SH,39.774873,39.830040,38.864630,38.947380,12354183,0
255970,2021-01-28,600597.SH,47.190201,49.243025,46.250355,46.423484,32409940,0


In [4]:
daily_price.query("sec_code=='600466.SH'")

,datetime,sec_code,open,high,low,close,volume,openinterest
0,2019-01-02,600466.SH,33.064891,33.496709,31.954503,32.386321,10629352,0
546,2019-01-03,600466.SH,32.262944,32.941515,31.399309,31.831127,8602646,0
1211,2019-01-04,600466.SH,31.399309,33.558397,31.337621,33.496709,12768116,0
1700,2019-01-07,600466.SH,33.496709,34.360344,33.373332,33.620085,10584321,0
2136,2019-01-08,600466.SH,33.311644,34.113591,32.694762,33.743462,10012902,0
...,...,...,...,...,...,...,...,...
253953,2021-01-22,600466.SH,30.245430,30.312942,29.502796,29.772845,17184055,0
253973,2021-01-25,600466.SH,29.570309,29.570309,28.827675,28.962699,23646174,0
254574,2021-01-26,600466.SH,28.962699,29.232748,28.692651,28.760163,9963442,0
255079,2021-01-27,600466.SH,28.760163,29.232748,28.692651,28.895187,12929331,0


In [68]:
# 筛选 600466.SH 和 603228.SH 2只股票的数据集
data1 = daily_price.query(f"sec_code=='600466.SH'").set_index('datetime').drop(columns=['sec_code'])
data2 = daily_price.query(f"sec_code=='603228.SH'").set_index('datetime').drop(columns=['sec_code'])
data2

,open,high,low,close,volume,openinterest
datetime,,,,,,
2019-01-02,50.660230,51.458513,50.394136,51.120778,426147,0
2019-01-03,50.609059,51.049137,50.107573,50.639762,492071,0
2019-01-04,50.199683,51.171950,49.278588,50.455543,665486,0
2019-01-07,50.864918,51.417575,50.353199,50.967262,689444,0
2019-01-08,51.110544,52.174920,50.250855,50.527183,931211,0
...,...,...,...,...,...,...
2021-01-22,59.955312,59.955312,57.478496,58.174461,6410959,0
2021-01-25,58.113052,58.133522,57.110045,57.498966,3445027,0
2021-01-26,57.498966,57.498966,54.305716,55.124498,7340180,0


In [90]:
cerebro = bt.Cerebro() # 大脑cerebro实例化

In [91]:
datafeed1 = bt.feeds.PandasData(dataname=data1, fromdate=datetime.datetime(2019,1,2), todate=datetime.datetime(2021,1,28))
cerebro.adddata(datafeed1, name='600466.SH')
datafeed2 = bt.feeds.PandasData(dataname=data2, fromdate=datetime.datetime(2019,1,2), todate=datetime.datetime(2021,1,28))
cerebro.adddata(datafeed2, name='603228.SH')

In [92]:
class MyStrategy(bt.Strategy):
    
    def __init__(self):
        pass

In [93]:
result = cerebro.run()
result = result[0] # 获取策略实例
data = results.data2 # 获取数据源

In [94]:
len(data)

506

In [96]:
# cerebro内的data0的名字
print(result.datas)
print(result.data._name)
print(result.data0._name)
print(result.data1._name)
# datafeed1的PandasData名字
print(datafeed1._name)

[<backtrader.feeds.pandafeed.PandasData object at 0x7fd2b9580a60>, <backtrader.feeds.pandafeed.PandasData object at 0x7fd2b95840d0>]
600466.SH
600466.SH
603228.SH
600466.SH


In [100]:
# lines的名称
print(result.lines.getlinealiases())
print(result.data0.lines.getlinealiases())
# 打印第一个表格的lines
print(result.datas[0].lines.getlinealiases())

('datetime',)
('close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime')
('close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime')


In [101]:
# 计算第一个数据集的s收盘价的20日均线，返回一个 Data feed
result.sma = bt.indicators.SimpleMovingAverage(result.datas[0].close, period=20)
print("--------- 打印 indicators 对象的 lines ----------")
print(result.sma.lines.getlinealiases())
print("---------- 直接打印 indicators 对象的所有 lines -------------")
print(result.sma.lines) 
print("---------- 直接打印 indicators 对象的第一条 lines -------------")
print(result.sma.lines[0])

--------- 打印 indicators 对象的 lines ----------
('sma',)
---------- 直接打印 indicators 对象的所有 lines -------------
---------- 直接打印 indicators 对象的第一条 lines -------------


'600466.SH'

In [65]:
data.getlinealiases()

('close', 'low', 'high', 'open', 'volume', 'openinterest', 'datetime')

In [72]:
data.lines.close[0]

54.91980265

In [83]:
datafeed1.lines.close[-1]

28.89518736

In [45]:
data.line.datetime

<bound method LineBuffer.datetime of <backtrader.linebuffer.LineBuffer object at 0x7fd2b90e2cd0>>

In [38]:
for i in range(len(data)):
    print(data.open[i]) # 打印开盘价
    print(data.high[i]) # 打印最高价
    print(data.low[i]) # 打印最低价
    print(data.close[i]) # 打印收盘价
    print(data.volume[i]) # 打印成交量
    print("-----------------------")

28.82767524
28.82767524
28.55762676
28.76016312
12826007.0
-----------------------


IndexError: array index out of range